**Como está a receita recorrente e inadimplência das assinaturas de fornecedores na plataforma?**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TABLE workspace.silver.charge AS
SELECT
    id,
    fk_subscription,
    amount,
    payment_method,
    status,
    payment_date
FROM solaria.public.charge
WHERE amount > 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY payment_date DESC NULLS LAST) = 1;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE OR REPLACE TABLE workspace.gold.dim_tempo_financeiro AS
SELECT DISTINCT
    CAST(date_trunc('day', payment_date) AS DATE) AS data_id,
    YEAR(payment_date) AS ano,
    MONTH(payment_date) AS mes,
    QUARTER(payment_date) AS trimestre
FROM workspace.silver.charge
WHERE payment_date IS NOT NULL;

CREATE OR REPLACE TABLE workspace.gold.dim_plano AS
SELECT id AS plano_id, name, value, cycle
FROM solaria.public.company_plans;

CREATE OR REPLACE TABLE workspace.gold.dim_fornecedor AS
SELECT
    s.id AS supplier_id,
    c.trade_name AS empresa,
    s.status,
    s.business_type
FROM solaria.public.supplier s
JOIN solaria.public.company c ON c.id = s.fk_company;

CREATE OR REPLACE TABLE workspace.gold.fato_cobrancas AS
SELECT
    ch.id AS charge_id,
    sub.fk_supplier AS supplier_id,
    sub.fk_plan AS plano_id,
    ch.amount,
    ch.payment_method,
    ch.status,
    CAST(date_trunc('day', ch.payment_date) AS DATE) AS data_id,
    ch.payment_date
FROM workspace.silver.charge ch
JOIN solaria.public.subscription sub ON sub.id = ch.fk_subscription;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.kpi_receita_mensal AS
SELECT
    dt.ano, dt.mes,
    SUM(f.amount) AS receita_mes,
    SUM(SUM(f.amount)) OVER (ORDER BY dt.ano, dt.mes) AS receita_acumulada
FROM workspace.gold.fato_cobrancas f
JOIN workspace.gold.dim_tempo_financeiro dt ON dt.data_id = f.data_id
WHERE f.status = 'PAID'
GROUP BY dt.ano, dt.mes;

CREATE OR REPLACE TABLE workspace.gold.kpi_distribuicao_status AS
SELECT
    status,
    COUNT(*) AS quantidade,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS percentual
FROM workspace.gold.fato_cobrancas
GROUP BY status;

CREATE OR REPLACE TABLE workspace.gold.kpi_receita_por_plano AS
SELECT
    dp.name AS plano,
    SUM(f.amount) AS receita_total,
    COUNT(*) AS total_cobrancas,
    RANK() OVER (ORDER BY SUM(f.amount) DESC) AS ranking
FROM workspace.gold.fato_cobrancas f
JOIN workspace.gold.dim_plano dp ON dp.plano_id = f.plano_id
WHERE f.status = 'PAID'
GROUP BY dp.name;